<a href="https://colab.research.google.com/github/itsayeshaqamar/flyrank-mlinternship-ayesha/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsayeshaqamar/flyrank-mlinternship-ayesha/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents one client-content page observation for one day. The daily performance table uses `report_date`, `client_hash_id`, and `content_hash_id` to define the observation grain.

For this assignment, I use March 2026 as the time window and use these daily observations as the basis for building page-level features for refresh prioritization.

**Time window:** March 1, 2026 to March 31, 2026.

In [3]:
%pip install -q duckdb huggingface_hub pandas

In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [6]:
import duckdb

con = duckdb.connect()

print("DuckDB connection created")

DuckDB connection created


In [7]:
con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face access configured")

Hugging Face access configured


In [9]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse path:", warehouse)

Warehouse path: hf://datasets/FlyRank/internship-warehouse


In [10]:
files = con.execute(f"""
    SELECT *
    FROM glob('{warehouse}/**/*.parquet')
    LIMIT 20
""").fetchdf()

files

,file
0,hf://datasets/FlyRank/internship-warehouse/dim...
1,hf://datasets/FlyRank/internship-warehouse/dim...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [11]:
# Show the complete list of warehouse files/tables

for i, path in enumerate(files):
    print(i, path)

0 file


In [15]:
print(type(files))
print(len(files))

<class 'pandas.core.frame.DataFrame'>
20


In [17]:
print(files.to_string(index=False))

                                                                                                  file
                                        hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
                                        hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance

In [19]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Number of rows:", len(df))

print("\nDate range:")
print("Start:", df["report_date"].min())
print("End:", df["report_date"].max())

print("\nUnique clients:", df["client_hash_id"].nunique())
print("Unique content pages:", df["content_hash_id"].nunique())

print("\nUnique client-page-day combinations:",
      df[["report_date", "client_hash_id", "content_hash_id"]].drop_duplicates().shape[0])

print("\nDuplicate client-page-day rows:",
      df.duplicated(
          subset=["report_date", "client_hash_id", "content_hash_id"]
      ).sum())

Number of rows: 9841378

Date range:
Start: 2026-03-01
End: 2026-03-31

Unique clients: 55
Unique content pages: 331437

Unique client-page-day combinations: 9841378

Duplicate client-page-day rows: 0


In [18]:

import pandas as pd

# March 2026 daily content performance
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df = pd.read_parquet(path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (9841378, 31)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

First 5 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Label / Proxy

There is no direct observed `refresh_needed` or `refresh_priority` label in this daily performance table.

Therefore, the ML-03 target remains a **proxy** rather than an observed outcome. A refresh-priority score can be defined later from observed performance and trend signals, but it should not be presented as a measured ground-truth label.

### Features

The performance fields used as candidate features are:

- GSC search performance: `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`
- GA4 performance: `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`
- Traffic sources: `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`
- AI traffic sources: `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`
- Engagement: `scroll_events`

These are observed performance signals that can support the refresh-prioritization task.

### Context

- `report_date` — observation date
- `client_hash_id` — anonymized client identifier
- `content_hash_id` — anonymized content identifier
- `client_has_gsc` — whether GSC is available for the client
- `client_has_ga4` — whether GA4 is available for the client
- `gsc_data_available` — whether GSC data is available for the observation
- `ga4_data_available` — whether GA4 data is available for the observation
- `month` — warehouse partition/month

These fields describe the observation, entity, or data availability rather than directly representing content performance.

### Excluded

`client_hash_id` and `content_hash_id` are excluded from model features because they identify entities rather than describing performance.

`report_date` and `month` are retained as context for time-window and partition checks but are not treated as direct performance features.

No raw field is used as the label because the warehouse does not contain an observed refresh decision or post-refresh outcome.

Data-availability fields are retained as context so that missing measurements are not automatically interpreted as zero performance.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the fields used in the data contract

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events"
]

context = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
    "month"
]

excluded = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month"
]

print("Number of feature fields:", len(features))
print("Number of context fields:", len(context))
print("Excluded fields:", excluded)

print("\nAll feature fields exist:",
      all(col in df.columns for col in features))

print("All context fields exist:",
      all(col in df.columns for col in context))

Number of feature fields: 23
Number of context fields: 8
Excluded fields: ['client_hash_id', 'content_hash_id', 'report_date', 'month']

All feature fields exist: True
All context fields exist: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification

The checks below verify the main contract claims: the observation grain, the March 2026 time window, the number of clients and content pages, duplicate client-page-day records, and missingness in the main performance fields.

The missing-value checks are important because GSC and GA4 coverage is not necessarily available for every client or observation. Missing values therefore need to be interpreted together with the corresponding data-availability fields rather than automatically treated as zero.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Verify the time window

print("TIME WINDOW CHECK")
print("-----------------")
print("Minimum report date:", df["report_date"].min())
print("Maximum report date:", df["report_date"].max())
print("Unique report dates:", df["report_date"].nunique())


# 2. Verify the client-page-day grain

print("\nGRAIN CHECK")
print("-----------")

total_rows = len(df)

unique_client_page_days = df[
    ["report_date", "client_hash_id", "content_hash_id"]
].drop_duplicates().shape[0]

duplicate_rows = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("Total rows:", total_rows)
print("Unique client-page-day combinations:", unique_client_page_days)
print("Duplicate client-page-day rows:", duplicate_rows)


# 3. Count clients and content pages

print("\nENTITY COUNTS")
print("-------------")
print("Unique clients:", df["client_hash_id"].nunique())
print("Unique content pages:", df["content_hash_id"].nunique())


# 4. Check missing values in important performance fields

check_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_ai",
    "scroll_events"
]

print("\nMISSING VALUE CHECK")
print("-------------------")

missing = df[check_fields].isna().sum()

missing_pct = (
    df[check_fields].isna().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct
})

display(missing_summary)


# 5. Check data availability flags

print("\nDATA AVAILABILITY")
print("-----------------")

print("GSC available:")
print(df["gsc_data_available"].value_counts(dropna=False))

print("\nGA4 available:")
print(df["ga4_data_available"].value_counts(dropna=False))

TIME WINDOW CHECK
-----------------
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31
Unique report dates: 31

GRAIN CHECK
-----------
Total rows: 9841378
Unique client-page-day combinations: 9841378
Duplicate client-page-day rows: 0

ENTITY COUNTS
-------------
Unique clients: 55
Unique content pages: 331437

MISSING VALUE CHECK
-------------------


,missing_count,missing_percent
gsc_impressions,0,0.00
gsc_clicks,0,0.00
gsc_avg_position,6230317,63.31
ga4_pageviews,3018741,30.67
ga4_sessions,3018741,30.67
ga4_users,3018741,30.67
ga4_engaged_sessions,3018741,30.67
sessions_organic,3018741,30.67
sessions_ai,3018741,30.67
scroll_events,3018741,30.67



DATA AVAILABILITY
-----------------
GSC available:
gsc_data_available
False    6230317
True     3611061
Name: count, dtype: int64

GA4 available:
ga4_data_available
False    6408671
None     3018741
True      413966
Name: count, dtype: int64


### Data Limits

This warehouse provides observed daily performance signals, but it does not directly record whether an editor decided to refresh a page or whether a refresh caused an improvement.

The March 2026 window contains up to 31 observed days per content page, but coverage is not balanced across all pages. Some pages have only one observed day, while others have observations for the full month.

GSC and GA4 coverage is also uneven. The observed availability rates show that performance measurements are not available for every observation, so missing values should not automatically be interpreted as zero activity.

Because the data contains multiple daily observations for the same content page, daily rows should not be treated as independent page-level examples. Page-level features should be aggregated carefully.

There is no observed `refresh_needed`, `refresh_priority`, or post-refresh outcome field in the available table. Therefore, any refresh-priority target remains a defined proxy rather than measured ground truth.

The data can support measured and directional comparisons of search, traffic, engagement, and AI-traffic signals. However, it cannot establish that refreshing a page will cause a particular future performance improvement.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantify some important data limitations

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("DATA LIMIT CHECKS")
print("==================")

# 1. Number of observed days per content page
days_per_page = (
    df.groupby("content_hash_id")["report_date"]
      .nunique()
)

print("Median observed days per content page:", days_per_page.median())
print("Minimum observed days per content page:", days_per_page.min())
print("Maximum observed days per content page:", days_per_page.max())


# 2. GSC and GA4 availability
gsc_rate = df["gsc_data_available"].mean() * 100
ga4_rate = df["ga4_data_available"].mean() * 100

print("\nGSC availability rate:", round(gsc_rate, 2), "%")
print("GA4 availability rate:", round(ga4_rate, 2), "%")


# 3. Check whether an observed refresh label exists
possible_labels = [
    col for col in df.columns
    if any(term in col.lower()
           for term in ["refresh_needed", "refresh_priority", "refreshed", "post_refresh"])
]

print("\nPossible direct label/outcome fields:")
print(possible_labels)

DATA LIMIT CHECKS
Median observed days per content page: 31.0
Minimum observed days per content page: 1
Maximum observed days per content page: 31

GSC availability rate: 36.69 %
GA4 availability rate: 6.07 %

Possible direct label/outcome fields:
[]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.